In [1]:
print("all ok")

all ok


In [2]:
import os
import asyncio

import pandas as pd

from dotenv import load_dotenv

from openai import AsyncOpenAI

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

from ragas.llms import llm_factory

from ragas.messages import ToolCall

from ragas.metrics.collections import (
    ToolCallAccuracy,
    ToolCallF1,
    AgentGoalAccuracyWithReference,
)

from langchain_core.tools import tool

from ragas.integrations.langgraph import convert_to_ragas_messages

d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY is missing. Please add it to the .env file."
    )

In [4]:
@tool
def search_flights(origin: str, destination: str) -> str:
    """
    Search flights between an origin and destination city.
    """

    return f"""
Available flights from {origin} to {destination}:

1. Emirates EK569
   Price: ₹28,000
   Type: Non-stop

2. Air India AI995
   Price: ₹24,500
   Type: 1 stop
""".strip()

In [5]:
@tool
def find_hotels(city: str, stars: int) -> str:
    """
    Find hotels in a city based on star rating.
    """

    return f"""
Available {stars}-star hotels in {city}:

1. Novotel {city}
   Price: ₹8,500 per night

2. Hilton Garden Inn {city}
   Price: ₹9,200 per night
""".strip()

In [6]:
@tool
def get_weather(city: str) -> str:
    """
    Get weather information for a city.
    """

    return f"""
Weather in {city}:

Temperature: 30°C
Condition: Sunny
""".strip()


In [7]:
# llm = ChatOpenAI(
#     model="gpt-5.6-luna",
#     temperature=0,
# )

# llm = ChatOpenAI(
#     model="gpt-5.6-luna",
#     use_responses_api=True,
#     output_version="responses/v1",
# )

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
)


In [8]:
llm.invoke("hi")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4c910a9acf', 'id': 'chatcmpl-EkXMvI1eErWjSKTvUZmyrv0S5z2Ae', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1e009-3a0e-7091-bb38-9f883f391800-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [9]:
agent = create_agent(
    model=llm,

    tools=[
        search_flights,
        find_hotels,
        get_weather,
    ],

    system_prompt="""
You are a travel planning assistant.

Use the available tools whenever the user asks for
flight, hotel, or weather information.

Important rules:

1. Use search_flights for flight requests.
2. Use find_hotels for hotel requests.
3. Use get_weather only when weather is requested.
4. Do not call unnecessary tools.
5. Use city names exactly as provided by the user.
6. After using the tools, provide a clear final answer.
"""
)

In [10]:
agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
Plan a Dubai trip from Bangalore.
Find flights from Bangalore to Dubai
and find a 4-star hotel in Dubai.
""".strip(),
        }
    ]
})

{'messages': [HumanMessage(content='Plan a Dubai trip from Bangalore.\nFind flights from Bangalore to Dubai\nand find a 4-star hotel in Dubai.', additional_kwargs={}, response_metadata={}, id='74989b46-9392-4aab-a1bc-ffe10ef8d94a'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 218, 'total_tokens': 273, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_936db6157f', 'id': 'chatcmpl-EkXMweqD9Uc1NdzKaPktaLzk1YCut', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1e009-4590-76ba-ab9b-d192c3e23243-0', tool_calls=[{'name': 'sea

In [11]:
openai_client = AsyncOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
)

evaluator_llm = llm_factory(
    "gpt-5.6-terra",
    client=openai_client,
)

In [12]:
tool_call_accuracy_metric = ToolCallAccuracy(strict_order=False)

In [13]:
tool_call_f1_metric = ToolCallF1()

In [14]:
agent_goal_accuracy_metric = AgentGoalAccuracyWithReference(llm=evaluator_llm)

In [15]:
dataset = [
    {
        "question": """
Plan a Dubai trip from Bangalore.
Find flights from Bangalore to Dubai
and find a 4-star hotel in Dubai.
""".strip(),

        "reference_tool_calls": [
            ToolCall(
                name="search_flights",
                args={
                    "origin": "Bangalore",
                    "destination": "Dubai",
                },
            ),
            ToolCall(
                name="find_hotels",
                args={
                    "city": "Dubai",
                    "stars": 4,
                },
            ),
        ],

        "reference_goal": """
The agent should provide flight options from Bangalore
to Dubai and 4-star hotel options in Dubai.
""".strip(),
    },

    {
        "question": """
Find flights from Bangalore to Singapore.
""".strip(),

        "reference_tool_calls": [
            ToolCall(
                name="search_flights",
                args={
                    "origin": "Bangalore",
                    "destination": "Singapore",
                },
            ),
        ],

        "reference_goal": """
The agent should provide flight options
from Bangalore to Singapore.
""".strip(),
    },

    {
        "question": """
Find a 5-star hotel in Mumbai.
""".strip(),

        "reference_tool_calls": [
            ToolCall(
                name="find_hotels",
                args={
                    "city": "Mumbai",
                    "stars": 5,
                },
            ),
        ],

        "reference_goal": """
The agent should provide 5-star hotel options
in Mumbai.
""".strip(),
    },
]

In [16]:
def get_message_text(message):

    content = message.content

    if isinstance(content, str):
        return content

    if isinstance(content, list):

        text_parts = []

        for block in content:

            if isinstance(block, str):
                text_parts.append(block)

            elif isinstance(block, dict) and "text" in block:
                text_parts.append(block["text"])

        return "\n".join(text_parts)

    return str(content)

In [17]:
def extract_tool_calls(ragas_trace):

    tool_calls = []

    for message in ragas_trace:

        message_tool_calls = getattr(
            message,
            "tool_calls",
            None,
        )

        if message_tool_calls:

            for tool_call in message_tool_calls:
                tool_calls.append(tool_call)

    return tool_calls


In [18]:
# async def run_agent(question):

#     result = await agent.ainvoke(
#         {
#             "messages": [
#                 {
#                     "role": "user",
#                     "content": question,
#                 }
#             ]
#         }
#     )

#     langchain_messages = result["messages"]

#     final_response = get_message_text(
#         langchain_messages[-1]
#     )

#     ragas_trace = convert_to_ragas_messages(
#         langchain_messages
#     )

#     actual_tool_calls = extract_tool_calls(
#         ragas_trace
#     )

#     return {
#         "response": final_response,
#         "ragas_trace": ragas_trace,
#         "actual_tool_calls": actual_tool_calls,
#     }

from ragas.messages import (
    HumanMessage as RagasHumanMessage,
    AIMessage as RagasAIMessage,
    ToolCall as RagasToolCall,
)


async def run_agent(question):

    result = await agent.ainvoke({
        "messages": [
            {
                "role": "user",
                "content": question,
            }
        ]
    })

    langchain_messages = result["messages"]

    final_response = get_message_text(
        langchain_messages[-1]
    )

    # ========================================================
    # EXTRACT TOOL CALLS DIRECTLY FROM LANGCHAIN MESSAGES
    # ========================================================

    actual_tool_calls = []

    for message in langchain_messages:

        tool_calls = getattr(
            message,
            "tool_calls",
            None,
        )

        if not tool_calls:
            continue

        for tool_call in tool_calls:

            actual_tool_calls.append(
                RagasToolCall(
                    name=tool_call["name"],
                    args=tool_call.get("args", {}),
                )
            )

    # ========================================================
    # CREATE RAGAS TRACE
    # ========================================================

    ragas_trace = [
        RagasHumanMessage(
            content=question,
        ),
        RagasAIMessage(
            content=final_response,
            tool_calls=actual_tool_calls,
        ),
    ]

    return {
        "response": final_response,
        "ragas_trace": ragas_trace,
        "actual_tool_calls": actual_tool_calls,
    }

In [19]:
# result = await agent.ainvoke({
#     "messages": [
#         {
#             "role": "user",
#             "content": """
# Plan a Dubai trip from Bangalore.
# Find flights from Bangalore to Dubai
# and find a 4-star hotel in Dubai.
# """.strip(),
#         }
#     ]
# })

# for message in result["messages"]:

#     print(type(message).__name__)

#     print(
#         "Tool Calls:",
#         getattr(message, "tool_calls", None),
#     )

#     print(
#         "Content:",
#         message.content,
#     )

#     print("-" * 100)

In [20]:
async def evaluate_test_case(row):

    question = row["question"]
    reference_tool_calls = row["reference_tool_calls"]
    reference_goal = row["reference_goal"]

    # ========================================================
    # RUN AGENT
    # ========================================================

    agent_result = await run_agent(question)

    response = agent_result["response"]
    ragas_trace = agent_result["ragas_trace"]
    actual_tool_calls = agent_result["actual_tool_calls"]

    # ========================================================
    # TOOL CALL ACCURACY
    # ========================================================

    tool_call_accuracy_result = await tool_call_accuracy_metric.ascore(
        user_input=ragas_trace,
        reference_tool_calls=reference_tool_calls,
    )

    # ========================================================
    # TOOL CALL F1
    # ========================================================

    tool_call_f1_result = await tool_call_f1_metric.ascore(
        user_input=ragas_trace,
        reference_tool_calls=reference_tool_calls,
    )

    # ========================================================
    # AGENT GOAL ACCURACY
    # ========================================================

    agent_goal_accuracy_result = await agent_goal_accuracy_metric.ascore(
        user_input=ragas_trace,
        reference=reference_goal,
    )

    # ========================================================
    # FORMAT TOOL CALLS
    # ========================================================

    actual_tool_calls_text = [
        f"{tool_call.name}({tool_call.args})"
        for tool_call in actual_tool_calls
    ]

    reference_tool_calls_text = [
        f"{tool_call.name}({tool_call.args})"
        for tool_call in reference_tool_calls
    ]

    # ========================================================
    # RETURN RESULT
    # ========================================================

    return {
        "question": question,
        "response": response,
        "actual_tool_calls": actual_tool_calls_text,
        "reference_tool_calls": reference_tool_calls_text,
        "tool_call_accuracy": tool_call_accuracy_result.value,
        "tool_call_f1": tool_call_f1_result.value,
        "agent_goal_accuracy": agent_goal_accuracy_result.value,
    }


In [21]:
async def run_evaluation():

    results = []

    for index, row in enumerate(dataset, start=1):

        print("\n" + "=" * 100)
        print(f"Running Test Case {index}/{len(dataset)}")
        print("=" * 100)

        result = await evaluate_test_case(row)

        results.append(result)

    return results

In [22]:
async def main():

    print("\n" + "=" * 100)
    print("RAGAS AGENTIC AI EVALUATION")
    print("=" * 100)

    # ========================================================
    # RUN EVALUATION
    # ========================================================

    results = await run_evaluation()

    # ========================================================
    # CONVERT TO DATAFRAME
    # ========================================================

    df = pd.DataFrame(results)

    # ========================================================
    # DETAILED RESULTS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED RESULTS")
    print("=" * 100)

    columns_to_show = [
        "question",
        "tool_call_accuracy",
        "tool_call_f1",
        "agent_goal_accuracy",
    ]

    print(df[columns_to_show].to_string(index=False))

    # ========================================================
    # SUMMARY
    # ========================================================

    average_tool_call_accuracy = df["tool_call_accuracy"].mean()
    average_tool_call_f1 = df["tool_call_f1"].mean()
    average_agent_goal_accuracy = df["agent_goal_accuracy"].mean()

    print("\n" + "=" * 100)
    print("SUMMARY")
    print("=" * 100)

    print(f"Average Tool Call Accuracy : {average_tool_call_accuracy:.2f}")
    print(f"Average Tool Call F1       : {average_tool_call_f1:.2f}")
    print(f"Average Agent Goal Accuracy: {average_agent_goal_accuracy:.2f}")

    # ========================================================
    # SAVE RESULTS
    # ========================================================

    output_file = "ragas_agent_evaluation_results.csv"

    df.to_csv(
        output_file,
        index=False,
    )

    print("\nResults saved to:")
    print(output_file)

    # ========================================================
    # PRINT EACH TEST CASE
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED AGENT RESULTS")
    print("=" * 100)

    for _, result_row in df.iterrows():

        print("\nQUESTION:")
        print(result_row["question"])

        print("\nAGENT RESPONSE:")
        print(result_row["response"])

        print("\nEXPECTED TOOL CALLS:")
        print(result_row["reference_tool_calls"])

        print("\nACTUAL TOOL CALLS:")
        print(result_row["actual_tool_calls"])

        print("\nTOOL CALL ACCURACY:")
        print(result_row["tool_call_accuracy"])

        print("\nTOOL CALL F1:")
        print(result_row["tool_call_f1"])

        print("\nAGENT GOAL ACCURACY:")
        print(result_row["agent_goal_accuracy"])

        print("\n" + "-" * 100)

In [23]:
await main()


RAGAS AGENTIC AI EVALUATION

Running Test Case 1/3

Running Test Case 2/3

Running Test Case 3/3

DETAILED RESULTS
                                                                                                  question  tool_call_accuracy  tool_call_f1  agent_goal_accuracy
Plan a Dubai trip from Bangalore.\nFind flights from Bangalore to Dubai\nand find a 4-star hotel in Dubai.                 1.0           1.0                  1.0
                                                                 Find flights from Bangalore to Singapore.                 1.0           1.0                  1.0
                                                                            Find a 5-star hotel in Mumbai.                 1.0           1.0                  1.0

SUMMARY
Average Tool Call Accuracy : 1.00
Average Tool Call F1       : 1.00
Average Agent Goal Accuracy: 1.00

Results saved to:
ragas_agent_evaluation_results.csv

DETAILED AGENT RESULTS

QUESTION:
Plan a Dubai trip from Bangalore.
Fi

## Deepeval agent Evaluation

In [24]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

from deepeval.test_case import (
    LLMTestCase,
    ConversationalTestCase,
    Turn,
    ToolCall,
)

from deepeval.metrics import (
    ToolCorrectnessMetric,
    ArgumentCorrectnessMetric,
    GoalAccuracyMetric,
)


In [25]:
@tool
def search_flights(origin: str, destination: str) -> str:
    """
    Search flights between an origin and destination city.
    """

    return f"""
Available flights from {origin} to {destination}:

1. Emirates EK569
   Price: ₹28,000
   Type: Non-stop

2. Air India AI995
   Price: ₹24,500
   Type: 1 stop
""".strip()


@tool
def find_hotels(city: str, stars: int) -> str:
    """
    Find hotels in a city based on star rating.
    """

    return f"""
Available {stars}-star hotels in {city}:

1. Novotel {city}
   Price: ₹8,500 per night

2. Hilton Garden Inn {city}
   Price: ₹9,200 per night
""".strip()


@tool
def get_weather(city: str) -> str:
    """
    Get weather information for a city.
    """

    return f"""
Weather in {city}:

Temperature: 30°C
Condition: Sunny
""".strip()


In [26]:
llm = ChatOpenAI(
    model="gpt-5.4-mini",
)

In [28]:
agent = create_agent(
    model=llm,
    tools=[
        search_flights,
        find_hotels,
        get_weather,
    ],
    system_prompt="""
You are a travel planning assistant.

Use the available tools whenever the user asks for
flight, hotel, or weather information.

Rules:

1. Use search_flights for flight requests.
2. Use find_hotels for hotel requests.
3. Use get_weather only when weather is requested.
4. Do not call unnecessary tools.
5. Use city names exactly as provided by the user.
6. After using the tools, provide a clear final answer.
""",
)


In [29]:
dataset = [
    {
        "question": """
Plan a Dubai trip from Bangalore.
Find flights from Bangalore to Dubai
and find a 4-star hotel in Dubai.
""".strip(),

        "expected_tools": [
            ToolCall(
                name="search_flights",
                input_parameters={
                    "origin": "Bangalore",
                    "destination": "Dubai",
                },
            ),
            ToolCall(
                name="find_hotels",
                input_parameters={
                    "city": "Dubai",
                    "stars": 4,
                },
            ),
        ],
    },

    {
        "question": """
Find flights from Bangalore to Singapore.
""".strip(),

        "expected_tools": [
            ToolCall(
                name="search_flights",
                input_parameters={
                    "origin": "Bangalore",
                    "destination": "Singapore",
                },
            ),
        ],
    },

    {
        "question": """
Find a 5-star hotel in Mumbai.
""".strip(),

        "expected_tools": [
            ToolCall(
                name="find_hotels",
                input_parameters={
                    "city": "Mumbai",
                    "stars": 5,
                },
            ),
        ],
    },
]


In [30]:
def get_message_text(message):

    content = message.content

    if isinstance(content, str):
        return content

    if isinstance(content, list):

        text_parts = []

        for block in content:

            if isinstance(block, str):
                text_parts.append(block)

            elif isinstance(block, dict) and "text" in block:
                text_parts.append(block["text"])

        return "\n".join(text_parts)

    return str(content)


In [31]:
def extract_tool_calls(messages):

    tool_calls = []

    for message in messages:

        message_tool_calls = getattr(message, "tool_calls", None)

        if not message_tool_calls:
            continue

        for tool_call in message_tool_calls:

            tool_calls.append(
                ToolCall(
                    name=tool_call["name"],
                    input_parameters=tool_call.get("args", {}),
                )
            )

    return tool_calls

In [32]:
def format_tool_calls(tool_calls):

    return [
        f"{tool_call.name}({tool_call.input_parameters})"
        for tool_call in tool_calls
    ]


In [33]:
async def run_agent(question):

    result = await agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question,
                }
            ]
        }
    )

    messages = result["messages"]

    response = get_message_text(messages[-1])

    actual_tool_calls = extract_tool_calls(messages)

    return {
        "response": response,
        "actual_tool_calls": actual_tool_calls,
    }


In [34]:
async def evaluate_test_case(row):

    question = row["question"]
    expected_tools = row["expected_tools"]

    # ========================================================
    # RUN AGENT
    # ========================================================

    agent_result = await run_agent(question)

    response = agent_result["response"]
    actual_tool_calls = agent_result["actual_tool_calls"]

    # ========================================================
    # CREATE SINGLE-TURN TEST CASE
    # ========================================================

    agent_test_case = LLMTestCase(
        input=question,
        actual_output=response,
        tools_called=actual_tool_calls,
        expected_tools=expected_tools,
    )

    # ========================================================
    # CREATE TOOL CORRECTNESS METRIC
    # ========================================================

    tool_correctness_metric = ToolCorrectnessMetric(
        threshold=0.7,
        include_reason=True,
    )

    # ========================================================
    # CREATE ARGUMENT CORRECTNESS METRIC
    # ========================================================

    argument_correctness_metric = ArgumentCorrectnessMetric(
        threshold=0.7,
        model="gpt-5.4",
        include_reason=True,
        async_mode=False,
    )

    # ========================================================
    # TOOL CORRECTNESS EVALUATION
    # ========================================================

    tool_correctness_metric.measure(agent_test_case)

    # ========================================================
    # ARGUMENT CORRECTNESS EVALUATION
    # ========================================================

    argument_correctness_metric.measure(agent_test_case)

    # ========================================================
    # CREATE CONVERSATIONAL TEST CASE
    # ========================================================

    conversation_test_case = ConversationalTestCase(
        turns=[
            Turn(
                role="user",
                content=question,
            ),
            Turn(
                role="assistant",
                content=response,
                tools_called=actual_tool_calls,
            ),
        ]
    )

    # ========================================================
    # CREATE GOAL ACCURACY METRIC
    # ========================================================

    goal_accuracy_metric = GoalAccuracyMetric(
        threshold=0.7,
        model="gpt-5.4",
        include_reason=True,
        async_mode=False,
    )

    # ========================================================
    # GOAL ACCURACY EVALUATION
    # ========================================================

    goal_accuracy_metric.measure(conversation_test_case)

    # ========================================================
    # RETURN RESULT
    # ========================================================

    return {
        "question": question,
        "response": response,
        "expected_tools": format_tool_calls(expected_tools),
        "actual_tool_calls": format_tool_calls(actual_tool_calls),

        "tool_correctness": tool_correctness_metric.score,
        "tool_correctness_reason": tool_correctness_metric.reason,

        "argument_correctness": argument_correctness_metric.score,
        "argument_correctness_reason": argument_correctness_metric.reason,

        "goal_accuracy": goal_accuracy_metric.score,
        "goal_accuracy_reason": goal_accuracy_metric.reason,
    }


In [35]:
async def run_evaluation():

    results = []

    for index, row in enumerate(dataset, start=1):

        print("\n" + "=" * 100)
        print(f"Running Test Case {index}/{len(dataset)}")
        print("=" * 100)

        result = await evaluate_test_case(row)

        results.append(result)

    return results

In [36]:
async def main():

    print("\n" + "=" * 100)
    print("DEEPEVAL AGENTIC AI EVALUATION")
    print("=" * 100)

    # ========================================================
    # RUN EVALUATION
    # ========================================================

    results = await run_evaluation()

    # ========================================================
    # CONVERT TO PANDAS
    # ========================================================

    df = pd.DataFrame(results)

    # ========================================================
    # DETAILED RESULTS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED RESULTS")
    print("=" * 100)

    columns_to_show = [
        "question",
        "tool_correctness",
        "argument_correctness",
        "goal_accuracy",
    ]

    print(df[columns_to_show].to_string(index=False))

    # ========================================================
    # CALCULATE AVERAGE SCORES
    # ========================================================

    average_tool_correctness = df["tool_correctness"].mean()
    average_argument_correctness = df["argument_correctness"].mean()
    average_goal_accuracy = df["goal_accuracy"].mean()

    # ========================================================
    # SUMMARY
    # ========================================================

    print("\n" + "=" * 100)
    print("SUMMARY")
    print("=" * 100)

    print(f"Average Tool Correctness     : {average_tool_correctness:.2f}")
    print(f"Average Argument Correctness : {average_argument_correctness:.2f}")
    print(f"Average Goal Accuracy        : {average_goal_accuracy:.2f}")

    # ========================================================
    # SAVE RESULTS
    # ========================================================

    output_file = "deepeval_agent_evaluation_results.csv"

    df.to_csv(output_file, index=False)

    print("\nResults saved to:")
    print(output_file)

    # ========================================================
    # PRINT DETAILED RESULTS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED AGENT EVALUATION REASONS")
    print("=" * 100)

    for _, result_row in df.iterrows():

        print("\nQUESTION:")
        print(result_row["question"])

        print("\nAGENT RESPONSE:")
        print(result_row["response"])

        print("\nEXPECTED TOOLS:")
        print(result_row["expected_tools"])

        print("\nACTUAL TOOL CALLS:")
        print(result_row["actual_tool_calls"])

        print("\nTOOL CORRECTNESS:")
        print(result_row["tool_correctness"])

        print("\nTOOL CORRECTNESS REASON:")
        print(result_row["tool_correctness_reason"])

        print("\nARGUMENT CORRECTNESS:")
        print(result_row["argument_correctness"])

        print("\nARGUMENT CORRECTNESS REASON:")
        print(result_row["argument_correctness_reason"])

        print("\nGOAL ACCURACY:")
        print(result_row["goal_accuracy"])

        print("\nGOAL ACCURACY REASON:")
        print(result_row["goal_accuracy_reason"])

        print("\n" + "-" * 100)

In [37]:
await main()


DEEPEVAL AGENTIC AI EVALUATION

Running Test Case 1/3


d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


Running Test Case 2/3


d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


Running Test Case 3/3


d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


DETAILED RESULTS
                                                                                                  question  tool_correctness  argument_correctness  goal_accuracy
Plan a Dubai trip from Bangalore.\nFind flights from Bangalore to Dubai\nand find a 4-star hotel in Dubai.               1.0                   1.0          0.625
                                                                 Find flights from Bangalore to Singapore.               1.0                   1.0          0.500
                                                                            Find a 5-star hotel in Mumbai.               1.0                   1.0          0.625

SUMMARY
Average Tool Correctness     : 1.00
Average Argument Correctness : 1.00
Average Goal Accuracy        : 0.58

Results saved to:
deepeval_agent_evaluation_results.csv

DETAILED AGENT EVALUATION REASONS

QUESTION:
Plan a Dubai trip from Bangalore.
Find flights from Bangalore to Dubai
and find a 4-star hotel in Dubai.

AGENT RE